# Keyword Extraction of Wikipedia Articles
Wikipedia is an encyclopedia that covers a large amount of diverse topics. All articles are created, corrected and updated by individuals. The goal is to correctly document as many topics as possible by collecting the knowledge of a large number of people. However, some articles stand out due to their completeness, scope and presentation, and for this they are marked with the distinction of the Excellent Article. 

As part of the Natural Language Processing lecture, a classification of Wikipedia articles is to be carried out as a sub-task of an assignment with the goal of being able to identify excellent articles. This notebook contains the code to accomplish this goal and is structured as follows:

- [1. Imports](article_classification.ipynb#1-imports)
- [2. Automated Data Check](article_classification.ipynb#2-check-data-availability)
- [3. Load and Process the Data](article_classification.ipynb#3-data-processing)
- [4. Train Neural Network]()
	- [4.1 Train/Validation/Test-Split](#thema1)
	- [4.2 Pre-Process the Texts](#thema2)
	- [4.3 Create Neural Network](#thema3)
	- [4.4 Train the Neural Network]()
	- [4.5 Validation of Results]()
- [5. Conclusion](#schluss)

## 1. Imports
Import the requiered libraties into the notebook.
If some libraries are not installed, you can use the `requierements.txt` and run
```
$ pip install -r requirements.txt
```
in the terminal.

In [ ]:
# Automated Data Download & Extraction
import os
import subprocess

# Effecient Processing of Wikipedia Dump
import mwxml # Import mwxml for effecient iteration of dump
import mwparserfromhell # Extract Features from Text

# Data Storage
import re # Import RegEx
from tqdm import tqdm # Import Progress-Bar
import pandas as pd
import numpy as np

# Pre-Processing
from sklearn.utils import resample
import spacy
from tqdm import tqdm
import nltk


# Neural Network

# Evaluation


## 2. Check Data Availability
In order to get the training data, a backup of the current wikipedia encyclopedie is needed. <br>
These dumps can be downloaded in every language by changing the url to https://dumps.wikimedia.org/[`insert_language (e.g. de, en)`]wiki/latest/. <br>
To make this example easy to run and have an equal data foundation, the download was automated with this cell. If the correct file already exists in the data folder, the download will be skipped. <br>
The file will be donwloaded as an `.bz2`-Archive and must be extracted before use. <br>
<b>Note: Due to the size of the file, the download may take a longer time depending on your internet connection.</b>

In [ ]:
if(not(os.path.exists("../Data"))):
   os.makedirs('../Data')

if(not(os.path.exists("../Data/dewiki-latest-pages-articles-multistream.xml"))):
    print("Articles-File not found. Download started... (this might take a while):")
    subprocess.call(['sh', '../bin/download-and-unzip-data.sh'])
print("Articles-File available at: ../Data/dewiki-latest-pages-articles-multistream.xml")

## 3. Data Processing
A big challenge is the effecient processing of the wikipedia dump files. Due to its size (ca. 26,8 GB), it is not possible to load the whole file into the memory. There are different approaches to deal with this problem but we chose to use the `mwxml`-library, which creates an generator-object that returns a single Wikipedia article at a time. In order to use natural language processing for the classification of the individual articles, the respecitve texts must be extracted. However, this poses another challenge due to the HTML-formatting.

In [ ]:
def extract_excellent_articles_to_dataframe(path_to_dump:str) -> pd.DataFrame:
    """
    Creates a dataset out of a wikipedia dump that contains only excellent articles.


    Parameters
    ----------
    path_to_dump : str. Relative file path to the unzipped wikipedia dump

    Returns
    -------
    df: pd.DataFrame. Pandas DataFrame that contains the processed wikipedia articles
    """
    data_index = [] # Empty array for the article ids
    label_index = [] # Empty array for the labels
    dump = mwxml.Dump.from_file(open(path_to_dump)) # Load Wikipedia dump and create generator object
    i = 0 # Set iteration variable to zero
    print("Step 1: Search Wikipedia Dataset")
    pbar = tqdm(total=5425758) # Create statusbar
    for page in dump: # Iterate over pages in dump
        for revisions in page: # Iterate over revisions of page
            try:
                if("Liste von Autoren" not in revisions.page.title): # Exclude non-article revisions
                    if(re.search(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", revisions.text)): # If article is marked as excelennt
                        label_index.append(1) # Add 1 (positive) as label to array
                    else:
                        label_index.append(0) # Add 0 (negative) as label to array
                    data_index.append(revisions.page.id) # Add page id to array
                    i += 1 # Increment iteration variable
                    pbar.update(1) # Updata status bar
            except Exception as e:
                print(e)
    pbar.close() # Stop progress bar
    print("Step 2: Resample Collected Data")
    temp_df = pd.DataFrame({'text_id': data_index, 'label': label_index}, columns=['text_id', 'label']) #  Create temporal dataframe
    valid_list = temp_df[temp_df['label'] == 1].values # Create list with valid article ids
    print("Step 3: Create Balanced Dataset")
    text = np.memmap('../Data/article_text.dat', dtype='object', mode='w+', shape=(len(valid_list), 1)) # Create memmap for texts
    dump = mwxml.Dump.from_file(open(path_to_dump)) # Load Wikipedia dump and create generator object
    i = 0 # Reset iteration variable to zero
    pbar2 = tqdm(total=len(valid_list)) # Create statusbar
    for page in dump: # Iterate over pages in dump
        for revisions in page: # Iterate over revisions in page
            try:
                if("Liste von Autoren" not in revisions.page.title): # Exclude non-article revisions
                    if revisions.page.id in valid_list: # If article is in resampled scope
                        temp_text = mwparserfromhell.parse(revisions.text) # Parse text
                        if any((re.match(r"{{Exzellent[|](\d*).(\D*)(\d*)[|](\d*)}}", str(template)) for template in temp_text.filter_templates())): # If article is markes as excellent
                            text[i] = re.sub(r"Exzellent (\d*).(\D*).(\d*)", "", (" ".join((" ".join(list(map(str, temp_text.filter_text())))).split()))) # Add cleaned text to memmap
                        i += 1 # Increment iteration variable
                        pbar2.update(1) # Update status bar
            except Exception as e:
                print(e)
        if(i>=len(valid_list)): # If valid list is completed
            break # Break iteration
    pbar2.close() # Close status bar
    return np.array(text).ravel() 

In [ ]:
url_to_dump = "../Data/dewiki-latest-pages-articles-multistream.xml"
dataframe = extract_excellent_articles_to_dataframe(path_to_dump=url_to_dump)

# Pickel dataframe to avoid running the processing function every time
#dataframe.to_pickle("../Data/processed_dataset.pkl")

In [ ]:
dataframe = pd.read_pickle("../Data/processed_dataset.pkl")

In [ ]:
dataframe = dataframe[dataframe["label"]==1]
docs = dataframe["text"].reset_index(drop=True)
docs

# Preprocessing

### Get document speficiy Stopwords

TF–IDF - term frequency–inverse document frequency

TfidfVectorizer class from scikit-learn to calculate the TF-IDF scores for the documents.

Calculate the TF-IDF scores:
Count the term frequency (TF): Calculate the number of times each term appears in a document.
Calculate the inverse document frequency (IDF): Measure the rarity of a term across the entire document collection.
Compute the TF-IDF score: Multiply the term frequency (TF) by the inverse document frequency (IDF) for each term in each document.

Select the top N words with the highest TF-IDF scores as potential stopwords.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Select the top N words with the highest TF-IDF scores as potential stopwords

vectorizer = TfidfVectorizer() # TfidfVectorizer object from sklearn

tfidf_matrix = vectorizer.fit_transform(docs) # Fit the vectorizer to the documents

feature_names = vectorizer.get_feature_names_out() # Get the names of the tokens

avg_tfidf_scores = tfidf_matrix.mean(axis=0).tolist()[0] # Get average TF-IDF score for each word across all documents

word_scores = list(zip(feature_names, avg_tfidf_scores)) # Create a list of words and their TF-IDF scores

word_scores.sort(key=lambda x: x[1], reverse=True) # Sort the word_scores in descending order

N = 250  # Number of stopwords to select
stopwords = [word for word, score in word_scores[:N]] # Potential stopwords

stopwords

### Create final Stopword List

Mithilfe der spacy STOP_WORDS Liste werden alle Wörter ohne Keyword aussgekraft entfernt. Da die Wikipediatext weitere Wörter enthalten die zur Formatierung des Textes verwendet werden (z.B ref), werden zuvor identifizierten Stopwords an dieser Stelle hinzugefügt.

In [ ]:
# Add potential stopwords to existing spacy stopwords list

from nltk.tokenize import RegexpTokenizer
from spacy.lang.de.stop_words import STOP_WORDS

original_stopword_len = len(STOP_WORDS)

stop_words = STOP_WORDS.copy()

stop_words |= set(stopwords) # Add the additional wikipedia specific stopwords

merged_stopword_len = len(stop_words)

print(f"{merged_stopword_len-original_stopword_len} document specific stopwords have been added to the spacy stopwords list")

stop_words

### Tokenizer

Der RegexpTokenizer teilt den Text eines Dokumentes in einzelne Wörter (Tokens) auf. Satzzeichen und Zahlen werden ebenfalls zu Tokens. Satzzeichen werden entfernt indem alle Tokens die nur ein Zeichen habe entfernt werden. Tokens die Zahlen enthalten werden ebenfalls entfernt.

In [ ]:
docs = docs[:1000]

In [ ]:
# Tokenize the documents

tokenizer = RegexpTokenizer(r'\w+')
for idx in range(len(docs)):
    docs[idx] = docs[idx].lower()  # To lowercase 
    docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words

# Remove numbers unnecessary words or signs

docs = [[token for token in doc if not token.isnumeric()] for doc in docs] # Remove numbers

docs = [[token for token in doc if len(token) > 1] for doc in docs] # Remove tokens with one character

docs

### Lemmatizizer

Während ein Stemmer Wörter auf ihren Stamm reduziert, indem die Endungen entfernt werden, wird ein Lemmatizer eingesetzt um Wörter in ihre tatsächliche Grundform zu bringt. Zwar ist der Stemmer einfacher und schneller, doch in diesem Fall sollen die Wörter weiterhin ein gramatikalisch sundvolles Wort ergeben, welches als Keyword verwendet werden kann. Aus Experte und Expertin soll nicht Expert (Stemmer) sonder Experte (Lemmatizer) werden.

- > Schnelligkeit _> Spacy ist am schnellsten weil https://spacy.io/api/lemmatizer just uses lookup tables and the only upstream task it relies on is POS tagging, so it should be relatively fast. For large amounts of text, SpaCy recommends using nlp.pipe, which can work in batches and has built in support for multiprocessing (with the n_process keyword), rather than than simply nlp.

In [ ]:
# Lemmatize tokens
test = docs[:2]

lemmatizer = spacy.load('de_core_news_sm')
test  = [[lemmatizer(token) for token in doc] for doc in test ]
test 

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

docs = test

for doc in nlp.pipe(docs, batch_size=32, n_process=3, disable=["parser", "ner"]):
    print([tok.lemma_ for tok in doc])

### Remove Stopwords

Apply stopword

In [ ]:
# Remove stopwords
len(docs)

docs = [[token for token in doc if token not in stop_words] for doc in docs]

docs

In [ ]:
# Lemmatize tokens
test = docs[:2]

lemmatizer = spacy.load('de_core_news_sm')
test  = [[lemmatizer(token) for token in doc] for doc in test ]
test 

# Keyword Extraction

## Latent Dirichlet Allocation (LDA) for Topic Modelling

https://radimrehurek.com/gensim/auto_examples/tutorials/run_lda.html

In [ ]:
# Compute bigrams
from gensim.models import Phrases

# Add bigrams and trigrams to docs (only ones that appear 20 times or more)

bigram = Phrases(docs, min_count=20)
for idx in range(len(docs)):
    for token in bigram[docs[idx]]:
        if '_' in token:
            # Token is a bigram, add to document
            docs[idx].append(token)

# Remove rare and common tokens

from gensim.corpora import Dictionary

# Create a dictionary representation of the documents

dictionary = Dictionary(docs)

# Filter out words that occur less than 20 documents, or more than 50% of the documents

dictionary.filter_extremes(no_below=20, no_above=0.5)

# Bag-of-words representation of the documents

corpus = [dictionary.doc2bow(doc) for doc in docs]

print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

In [ ]:
from gensim.models import LdaModel

# Set training parameters
num_topics = 10
chunksize = 2000
passes = 20
iterations = 400
eval_every = None  # Don't evaluate model perplexity, takes too much time.

# Make an index to word dictionary
temp = dictionary[0] 
id2word = dictionary.id2token

model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every
)

In [ ]:
keywords_list = []

for i in range (0, len(corpus)):
    topic_distribution = model.get_document_topics(corpus[i])
    # Sort the topics by their probability in descending order
    sorted_topics = sorted(topic_distribution, key=lambda x: x[1], reverse=True)
    # Get the top N keywords for the highest-probability topic
    top_keywords = model.show_topic(sorted_topics[0][0], topn=3)
    # Extract and return the keyword strings
    keywords = [keyword for keyword, _ in top_keywords]
    keywords_list.append(keywords)

print(keywords_list)

## Yet Another Keyword Extractor (Yake)

https://github.com/LIAAD/yake

In [ ]:
import yake

# ! pip install git+https://github.com/LIAAD/yake

language = "de"
max_ngram_size = 1
deduplication_threshold = 0.9
deduplication_algo = 'seqm'
windowSize = 1
numOfKeywords = 3


for i in docs :

    '''str = []
    for j in i:
        str.append(j.text)
    sentence = ' '.join(str)'''

    str = []
    for j in i:
        str.append(j)
    sentence = ' '.join(str)

    custom_kw_extractor = yake.KeywordExtractor(lan=language, n=max_ngram_size, dedupLim=deduplication_threshold, dedupFunc=deduplication_algo, windowsSize=windowSize, top=numOfKeywords, features=None)
    keywords = custom_kw_extractor.extract_keywords(sentence)

    keyword_list = []

    for kw in keywords:
        keyword_list.append(kw[0]) # select Keyword without score
    
    print(keyword_list)

## KeyBert

https://github.com/MaartenGr/KeyBERT: 

KeyBert verwendet BERT-Embeddings und die Cosine-Similarity, um Keywords in einem Dokument zu finden, die das Dokument am besten zusammenfassen.

Dafür werden Document-Embeddings mit BERT extrahiert, um eine Darstellung auf Dokumentenebene zu erhalten. Dann werden Worteinbettungen für N-Gramm-Wörter/Phrasen extrahiert. Schließlich wird die Cosine-Similarity verwendet, um die Wörter/Phrasen zu finden, die dem Dokument am ähnlichsten sind. Die ähnlichsten Wörter können dann als die Wörter identifiziert werden, die das gesamte Dokument am besten beschreiben. Jedoch benötigen die BERT-Modelle viele Ressourcen, wenn sie große Dokumente verarbeiten müssen.

In [ ]:
from keybert import KeyBERT

doc = """
         Supervised learning is the machine learning task of learning a function that
         maps an input to an output based on example input-output pairs. It infers a
         function from labeled training data consisting of a set of training examples.
         In supervised learning, each example is a pair consisting of an input object
         (typically a vector) and a desired output value (also called the supervisory signal).
         A supervised learning algorithm analyzes the training data and produces an inferred function,
         which can be used for mapping new examples. An optimal scenario will allow for the
         algorithm to correctly determine the class labels for unseen instances. This requires
         the learning algorithm to generalize from the training data to unseen situations in a
         'reasonable' way (see inductive bias).
      """
kw_model = KeyBERT('distilbert-base-nli-mean-tokens')
keywords = kw_model.extract_keywords(doc)

In [ ]:
keywords

In [ ]:
docs[1]

In [ ]:
# ! pip install keybert

test = docs[:3]

from keybert import KeyBERT

kw_model = KeyBERT()

for i in test :

    str = []
    for j in i:
        str.append(j)
    sentence = ' '.join(str)

    keywords = kw_model.extract_keywords(sentence, keyphrase_ngram_range=(1, 1), stop_words=None)
    print(keywords)

In [ ]:
for i in test :
    str = []
    for j in i:
        str.append(j)
    sentence = ' '.join(str)

In [ ]:
# Detokenize Tokens back into String for further Keyword Extractors

from nltk.tokenize.treebank import TreebankWordDetokenizer

detokenizer = TreebankWordDetokenizer()

for idx in range(len(docs)):
    docs[idx] = detokenizer.detokenize(docs[idx])  # Split into words

In [ ]:
import yake
from keybert import KeyBERT

test = docs[:5]

# Create candidates
kw_extractor = yake.KeywordExtractor(top=50)

# Pass candidates to KeyBERT
kw_model = KeyBERT()

for i in test :

    candidates = kw_extractor.extract_keywords(test)
    candidates = [candidate[0] for candidate in candidates]
    keywords = kw_model.extract_keywords(test, candidates=candidates)

keywords


pip install git+https://github.com/boudinfl/pke.git

In [ ]:
import pke

for i in docs :

    '''str = []
    for j in i:
        str.append(j.text)
    sentence = ' '.join(str)'''

    str = []
    for j in i:
        str.append(j)
    sentence = ' '.join(str)
    
    # initialize keyphrase extraction model, here TopicRank
    extractor = pke.unsupervised.FirstPhrases()

    # load text
    extractor.load_document(input=sentence, language='de')

    # keyphrase candidate selection, in the case of TopicRank: sequences of nouns
    # and adjectives (i.e. `(Noun|Adj)*`)
    extractor.candidate_selection()

    # candidate weighting, in the case of TopicRank: using a random walk algorithm
    extractor.candidate_weighting()

    # N-best selection, keyphrases contains the 10 highest scored candidates as
    # (keyphrase, score) tuples
    keyphrases = extractor.get_n_best(n=3, stemming=True)

    print(keyphrases)

# Evaluation

To evaluate the performance of topic modeling, googel keyword research is used. Googel is a search engine that identifies and analyzes similar keywords as topic modeling. Since googel is the most used search engine, the following section checks whether the correct Wikipedia link can be found using googel's keyword research.

The rank of the search results is used as a performance index.

Wenn keine Labels für die Evaluation einer Keyword Extraction-Methode vorhanden sind, können dennoch einige Methoden verwendet werden, um die Performance zu beurteilen. Hier sind einige Ansätze:

Unüberwachtes Vergleichen von Ergebnissen: Führen Sie die Keyword Extraction-Methode auf einer Stichprobe von Texten aus Ihrem Korpus durch und vergleichen Sie die extrahierten Keywords mit Ihren eigenen Erwartungen oder einer manuellen Überprüfung. Obwohl dies subjektiv sein kann, können Sie so einen Eindruck von der Qualität der extrahierten Keywords bekommen.
Vergleich mit existierenden Ressourcen: Es gibt viele etablierte Ressourcen für Keyword Extraction, wie zum Beispiel Lexika, Thesauri oder Wörterbücher. Vergleichen Sie die extrahierten Keywords mit den vorhandenen Ressourcen, um eine Vorstellung davon zu bekommen, wie gut Ihre Methode funktioniert.
Expertenbewertung: Bitten Sie Fachexperten, die extrahierten Keywords zu überprüfen und sie entsprechend ihrer Relevanz zu bewerten. Experten können in der Regel besser beurteilen, ob die extrahierten Keywords tatsächlich wichtige Begriffe sind.
Konsistenzüberprüfung: Wenden Sie Ihre Methode auf verschiedene Teilmengen desselben Korpus an und prüfen Sie, ob die extrahierten Keywords konsistent sind. Wenn die Methode robust ist, sollten ähnliche Keywords in verschiedenen Durchläufen gefunden werden.
Verwenden von Metriken für unüberwachtes Text-Mining: Es gibt einige Metriken, die speziell für die unüberwachte Evaluation von Text-Mining-Aufgaben entwickelt wurden. Ein Beispiel ist die Term Frequency-Inverse Document Frequency (TF-IDF), mit der die Relevanz von Begriffen in einem Textkorpus bewertet werden kann. Sie können diese Metriken verwenden, um die extrahierten Keywords zu bewerten und zu vergleichen.
Diese Methoden können zwar nicht die gleiche Genauigkeit wie eine überwachte Evaluation bieten, können aber dennoch eine Vorstellung von der Leistung einer Keyword Extraction-Methode geben. Es ist wichtig zu beachten, dass die Evaluation von Keyword Extraction immer mit gewissen Unsicherheiten verbunden ist, da es keinen eindeutigen "richtigen" Satz von Keywords gibt.

In [ ]:
from googlesearch import search
import time

for i in keywords_list[:1]:
    
    # Definie Search
    query = "site:de.wikipedia.org" + " ".join(i) # Wikipedia + identified Key Words

    # Get Search results
    url_results = []
    for url in search(query, num_results=20):
        url_results.append(url)
        #print(url)

    # Check if url leads to the right article and save index as performance score
    titel = "Aristoteles"
    performance_index = []
    if "https://de.wikipedia.org/wiki/"+titel in url_results:
        index = url_results.index("https://de.wikipedia.org/wiki/"+titel)
        print("Correct url has been found on the", index+1 , "search result")
        performance_index.append(index)
    else:
        index = np.nan
        performance_index.append(index)

    time.sleep(30)
    break